# تجربة CodeVision: كشف إطارات محرر الكود من فيديو واحد\n\n<div dir="rtl">\n\n## هدف التجربة\n\nنبني مصنفًا صغيرًا باستخدام **CNN وPyTorch** لتمييز إطار الفيديو الذي يعرض محرر/شيفرة برمجية (`code`) عن الإطار الذي لا يعرضه (`other`). نستخرج عينات زمنية من فيديو تعليمي واحد، ثم يراجع المتدرب صور العينات ويضع تسمياتها، وبعدها ندرب ونقيّم النموذج على أجزاء زمنية منفصلة من الفيديو.\n\nهذه تجربة عملية مبنية على درس **CNN (اليوم السادس)**: فحص الصور، تجهيز البيانات، بناء الشبكة، التدريب، منحنيات الخسارة والدقة، التقييم ومصفوفة الالتباس. لا ندّعي أن فيديو واحدًا يكفي لتعليم نموذج عام أو لاستخراج نص الكود نفسه؛ هذه التجربة تقيس فقط اكتشاف الإطارات المناسبة للاستخراج.\n\n### لماذا نبدأ بهذه التجربة؟\n\nفحصت خدمة التدريب في المشروع: `desktop/Services/TrainingLabService.cs` تحسب loss وaccuracy بمعادلات ثابتة، تنتظر 0.4 ثانية لكل حقبة، ثم تحفظ ملفًا نصيًا باسم أوزان `.pt` وسجلًا بدقة ثابتة. هذا **محاكاة وليس تدريب PyTorch فعليًا**. كذلك يحتوي `core/video_pipeline.py` مسارات احتياطية تنشئ مشهدًا ونصًا وشيفرة شكلية إذا لم ينتج الاستخراج نتيجة، وتكتب ثقة ثابتة `0.962`. لذلك لا نستخدم هذه المخرجات كدليل نجاح؛ سنقيس صورًا حقيقية وتسميات يراجعها الإنسان.\n\n> تنبيه: نتيجة تجربة فيديو واحد أولية ومحدودة بهذا الفيديو. لا تستخدمها لادعاء دقة عامة أو لاستبدال مراجعة الكود المستخرج يدويًا.\n\n</div>

## 0. إعداد البيئة والفيديو\n\nشغّل الخلايا بالترتيب. في Colab ارفع الفيديو عند طلبه. محليًا يمكنك ترك المسار الافتراضي أو تغييره.

In [ ]:
# في Colab غالبًا تكون PyTorch مثبتة مسبقًا؛ لا ننزّل TensorFlow.
%pip -q install opencv-python-headless pandas matplotlib scikit-learn pillow

In [ ]:
from pathlib import Path
import os, json, time, random, math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| الجهاز:', DEVICE)

# المسار الموجود عند صاحب المشروع؛ في Colab سترفع الخلية التالية الفيديو إذا لم يوجد.
VIDEO_PATH = Path(r'D:\Programming\دورات\فيديوهات برمجه\فيديوهات برمجه\Desing Patren\Creational Design Patterns بالعربى 2\5-_Singleton_Pattern_-_Lazy_Initialization_Implementation(1080p).mp4')
WORK_DIR = Path.cwd() / 'codevision_one_video_experiment'
FRAME_DIR = WORK_DIR / 'frames'
WORK_DIR.mkdir(parents=True, exist_ok=True); FRAME_DIR.mkdir(parents=True, exist_ok=True)

if not VIDEO_PATH.exists():
    try:
        from google.colab import files
        print('ارفع الفيديو المحدد: درس Singleton Lazy Initialization (حوالي 5 MB).')
        uploaded = files.upload()
        if not uploaded: raise RuntimeError('لم يُرفع فيديو.')
        name, content = next(iter(uploaded.items()))
        VIDEO_PATH = WORK_DIR / Path(name).name
        VIDEO_PATH.write_bytes(content)
    except ImportError:
        raw = input('اكتب المسار الكامل لملف الفيديو MP4: ').strip().strip('"')
        VIDEO_PATH = Path(raw)
if not VIDEO_PATH.is_file(): raise FileNotFoundError(f'لم أجد الفيديو: {VIDEO_PATH}')
print('الفيديو:', VIDEO_PATH.name, '| الحجم MB:', round(VIDEO_PATH.stat().st_size/1024**2, 1))

## 1. فحص الفيديو واستخراج إطارات حقيقية\n\nنأخذ صورة كل عدة ثوانٍ، بدل تحميل جميع إطارات الفيديو إلى الذاكرة. عيّنة الإطار مستقلة عن قرار الخوارزمية؛ سنرى الإطارات قبل وضع الملصقات. لا نحذف أي ملف أصلي.

In [ ]:
SAMPLE_EVERY_SEC = 2        # قلّلها إلى 3 لعينات أكثر، أو زدها لتسريع إعداد الملصقات
MAX_SAMPLES = 600          # حد يمنع امتلاء الذاكرة/المجلد

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened(): raise RuntimeError('تعذر فتح الفيديو بواسطة OpenCV.')
fps = cap.get(cv2.CAP_PROP_FPS) or 0
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0); height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
duration = frame_count / fps if fps > 0 else 0
print(f'الدقة {width}×{height} | FPS={fps:.2f} | عدد الإطارات={frame_count:,} | المدة={duration/60:.1f} دقيقة')

# امسح صور التجربة السابقة فقط، ولا تلمس ملف الفيديو أو مخرجات الاستديو.
for old in FRAME_DIR.glob('frame_*.jpg'): old.unlink()
records=[]
times = np.arange(0, duration, SAMPLE_EVERY_SEC)[:MAX_SAMPLES]
for sec in times:
    cap.set(cv2.CAP_PROP_POS_MSEC, float(sec*1000))
    ok, bgr = cap.read()
    if not ok: continue
    idx = len(records)
    name = f'frame_{idx:04d}_{sec:07.1f}s.jpg'
    path = FRAME_DIR / name
    cv2.imwrite(str(path), bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    # مؤشر OpenCV تقريبي مشابه للفكرة الموجودة في المحرك، وليس تصنيفًا مُتعلّمًا.
    edge_density = float((edges > 0).mean())
    dark_ratio = float((gray < 110).mean())
    code_score = min(1.0, 0.65*min(edge_density/0.12, 1.0) + 0.35*dark_ratio)
    records.append({'frame_id':idx,'timestamp_sec':round(float(sec),2),'image':name,
                    'edge_density':edge_density,'dark_ratio':dark_ratio,'opencv_score':code_score,'label':''})
cap.release()
manifest = pd.DataFrame(records)
if manifest.empty: raise RuntimeError('لم تُستخرج إطارات؛ تحقق من ترميز الفيديو.')
manifest.to_csv(WORK_DIR/'frames_to_label.csv',index=False,encoding='utf-8-sig')
print(f'استخرجنا {len(manifest)} إطارًا إلى {FRAME_DIR}')
display(manifest.head())

## 2. راجع العينات وضع الملصقات بنفسك\n\nافتح صورة الاتصال لكل مجموعة، ثم حدّث `frames_to_label.csv` في مجلد التجربة. اكتب `code` إذا ظهر محرر/شيفرة يمكن قراءة جزء منها، و`other` للمقدمة أو وجه المحاضر أو مشهد لا يحتوي على شاشة كود. لا تعتبر صورة فيها نافذة صغيرة غير مقروءة `code` إلا إذا كان هذا هو معيارك المعلن.\n\nفي Colab نزّل الملف، عدّله محليًا، ثم ارفعه في الخلية التالية. محليًا عدّل الملف مباشرة. هذه الخطوة تجعل الملصقات حقيقية بدل اختراع تسميات من خوارزمية نريد تقييمها.\n\nبعد كل مجموعة، استخدم رقم `frame_id` الظاهر في اسم الصورة لمطابقة الصف. حاول وضع أمثلة من الفئتين على امتداد الفيديو كله؛ استهدف 30 صورة على الأقل لكل فئة إن سمح محتوى الفيديو.

In [ ]:
from IPython.display import display, Image as IPyImage

def show_contact_sheet(start=0, count=32, cols=4):
    subset = manifest.iloc[start:start+count]
    if subset.empty: print('لا توجد عينات بهذا النطاق.'); return
    rows = math.ceil(len(subset)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows*3))
    axes = np.array(axes).reshape(-1)
    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = Image.open(FRAME_DIR/row.image).convert('RGB')
        img.thumbnail((420,240))
        ax.imshow(img); ax.axis('off')
        ax.set_title(f"ID {row.frame_id} | {row.timestamp_sec:.0f}s | OpenCV={row.opencv_score:.2f}",fontsize=9)
    for ax in axes[len(subset):]: ax.axis('off')
    plt.tight_layout(); plt.show()

# غيّر البداية 0 ثم 32 ثم 64... لاستعراض الفيديو كله.
show_contact_sheet(start=0,count=32)

In [ ]:
# في Google Colab: نزّل قالب الملصقات قبل تعبئته.
try:
    from google.colab import files
    files.download(str(WORK_DIR/'frames_to_label.csv'))
except ImportError:
    print('عدّل الملف frames_to_label.csv الموجود في:', WORK_DIR)

# بعد تعبئة عمود label بقيم code / other في Colab ارفع الملف المعدل هنا.
try:
    from google.colab import files
    uploaded_labels = files.upload()
    if uploaded_labels:
        label_name, label_bytes = next(iter(uploaded_labels.items()))
        label_path = WORK_DIR/'frames_labeled.csv'
        label_path.write_bytes(label_bytes)
    else: label_path = WORK_DIR/'frames_to_label.csv'
except ImportError:
    label_path = WORK_DIR/'frames_to_label.csv'

labels_df = pd.read_csv(label_path)
labels_df['label'] = labels_df['label'].fillna('').astype(str).str.strip().str.lower()
labels_df = labels_df[labels_df.label.isin(['code','other'])].copy()
print('عدد الملصقات الصالحة:',len(labels_df))
display(labels_df.label.value_counts().rename_axis('الفئة').to_frame('العدد'))
if labels_df.empty: raise ValueError('لا توجد ملصقات بعد. أكمل العمود label بـ code أو other ثم أعد هذه الخلية.')

## 3. التقسيم وتجهيز الصور\n\nحتى لا تتسرب إطارات متجاورة شبه متطابقة بين التدريب والاختبار، نقسم **زمنيًا**: أول 60% من مدة الفيديو للتدريب، ثم 20% للتحقق، وآخر 20% للاختبار. هذا تقسيم صارم لكنه يجعل التقييم أصعب وأكثر نزاهة. إذا غابت فئة عن قسم ما، ستظهر المشكلة قبل التدريب؛ عندها زد العينات أو راجع تسمياتك، ولا نختلق نتائج.\n\nنحوّل الصور إلى RGB بحجم `160×90` ونطبّع القيم إلى `[0,1]`. الشبكة صغيرة لتناسب تجربة تعليمية ووقت Colab.

In [ ]:
labels_df['split'] = pd.cut(labels_df.timestamp_sec,
    bins=[-np.inf, duration*0.60, duration*0.80, np.inf],
    labels=['train','validation','test'], include_lowest=True).astype(str)
print(pd.crosstab(labels_df['split'],labels_df['label']))
for split in ['train','validation','test']:
    present=set(labels_df.loc[labels_df.split==split,'label'])
    if present != {'code','other'}:
        raise ValueError(f"القسم {split} لا يحتوي الفئتين. أضف/راجع الملصقات في هذه الفترة الزمنية حتى نقيّم النموذج بإنصاف. الفئات الحالية: {present}")

class FrameDataset(Dataset):
    def __init__(self, frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row=self.frame.iloc[i]
        img=Image.open(FRAME_DIR/row.image).convert('RGB').resize((160,90))
        arr=np.asarray(img,dtype=np.float32)/255.0
        x=torch.from_numpy(arr).permute(2,0,1)
        y=0 if row.label=='other' else 1
        return x,torch.tensor(y,dtype=torch.long)

train_df=labels_df[labels_df.split=='train']; val_df=labels_df[labels_df.split=='validation']; test_df=labels_df[labels_df.split=='test']
BATCH_SIZE=32
train_loader=DataLoader(FrameDataset(train_df),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
val_loader=DataLoader(FrameDataset(val_df),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
test_loader=DataLoader(FrameDataset(test_df),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
xb,yb=next(iter(train_loader)); print('شكل batch:',tuple(xb.shape),'| labels:',tuple(yb.shape))

## 4. بناء CNN صغيرة\n\nكل كتلة `Conv2d → ReLU → MaxPool2d` تتعلم أنماطًا بصرية ثم تقلل أبعاد الصورة. في النهاية نستخدم متوسطًا مكانيًا وطبقة خطية تعطي درجتين: `other` و`code`. نستخدم `CrossEntropyLoss` لأنها مسألة تصنيف بفئتين وتسميات صحيحة.

In [ ]:
class FrameCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,96,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1))
        )
        self.classifier=nn.Sequential(nn.Flatten(),nn.Dropout(0.25),nn.Linear(96,2))
    def forward(self,x): return self.classifier(self.features(x))

model=FrameCNN().to(DEVICE)
counts=train_df.label.value_counts()
weights=torch.tensor([len(train_df)/ (2*counts.get('other',1)), len(train_df)/(2*counts.get('code',1))],dtype=torch.float32,device=DEVICE)
criterion=nn.CrossEntropyLoss(weight=weights)
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)
print(model)
print('المعاملات القابلة للتعلم:',sum(p.numel() for p in model.parameters() if p.requires_grad))

## 5. تدريب فعلي وتسجيل History\n\nهنا يحدث التدريب الحقيقي: `forward → loss → backward → optimizer.step`. نسجل في كل حقبة خسارة ودقة التدريب والتحقق، ونحتفظ بأفضل أوزان التحقق مع إيقاف مبكر. عدد الحقبات ليس هو مدة التدريب؛ الزمن يعتمد على عدد الصور والجهاز.

In [ ]:
def run_epoch(loader, training=False):
    model.train(training)
    total_loss=0.0; correct=0; seen=0
    for xb,yb in loader:
        xb=xb.to(DEVICE); yb=yb.to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits=model(xb); loss=criterion(logits,yb)
            if training: loss.backward(); optimizer.step()
        total_loss += loss.item()*len(yb)
        correct += (logits.argmax(1)==yb).sum().item(); seen += len(yb)
    return total_loss/max(seen,1), correct/max(seen,1)

EPOCHS=20; PATIENCE=4
history=[]; best_val=float('inf'); best_state=None; stale=0
start_train=time.perf_counter()
for epoch in range(1,EPOCHS+1):
    tr_loss,tr_acc=run_epoch(train_loader,training=True)
    va_loss,va_acc=run_epoch(val_loader,training=False)
    history.append({'epoch':epoch,'train_loss':tr_loss,'val_loss':va_loss,
                    'train_accuracy':tr_acc,'val_accuracy':va_acc,'seconds':time.perf_counter()-start_train})
    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {tr_loss:.4f}, acc {tr_acc:.3f} | val loss {va_loss:.4f}, acc {va_acc:.3f}")
    if va_loss < best_val:
        best_val=va_loss; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; stale=0
    else:
        stale += 1
        if stale>=PATIENCE: print('إيقاف مبكر: لم تتحسن خسارة التحقق.'); break
if best_state is None: raise RuntimeError('لم تُسجل أوزان صالحة.')
model.load_state_dict(best_state)
history_df=pd.DataFrame(history)
print(f'مدة التدريب الفعلية: {history_df.seconds.iloc[-1]:.1f} ثانية | أفضل val_loss={best_val:.4f}')

### قراءة سجل التدريب\n\nهبوط خسارة التدريب وحده لا يكفي. نقارن منحنى التحقق: إذا تحسنت خسارة التدريب بينما ساءت خسارة التحقق فقد بدأ فرط التكيّف. الفجوة الكبيرة بين train وvalidation تعني أن النموذج حفظ خصائص العينات بدل أن يعمم.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].plot(history_df.epoch,history_df.train_loss,label='train'); axes[0].plot(history_df.epoch,history_df.val_loss,label='validation')
axes[0].set_title('Loss عبر الحقب'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=.25); axes[0].legend()
axes[1].plot(history_df.epoch,history_df.train_accuracy,label='train'); axes[1].plot(history_df.epoch,history_df.val_accuracy,label='validation')
axes[1].set_title('Accuracy عبر الحقب'); axes[1].set_xlabel('Epoch'); axes[1].set_ylim(0,1); axes[1].grid(alpha=.25); axes[1].legend()
plt.tight_layout(); plt.show()
history_df.to_csv(WORK_DIR/'training_history.csv',index=False)

## 6. تقييم نهائي على جزء الاختبار\n\nلم نستخدم هذا القسم لاختيار الأوزان؛ نستخدمه مرة واحدة هنا للتقييم. نقارن أيضًا مؤشر OpenCV القديم على الصور نفسها. هذه مقارنة كشف «هل الإطار يحتوي شاشة كود؟» فقط، وليست قياس دقة OCR أو صحة ملفات C#. إذا كانت النتائج ضعيفة، فهذا اكتشاف مفيد: نحتاج بيانات أكثر أو طريقة استخراج/تسميات أفضل.

In [ ]:
@torch.no_grad()
def collect_predictions(loader):
    model.eval(); ys=[]; ps=[]; probs=[]
    for xb,yb in loader:
        logits=model(xb.to(DEVICE)); p=torch.softmax(logits,dim=1)[:,1].cpu().numpy()
        ys.extend(yb.numpy().tolist()); probs.extend(p.tolist()); ps.extend((p>=0.5).astype(int).tolist())
    return np.array(ys),np.array(ps),np.array(probs)

y_true,y_pred,y_prob=collect_predictions(test_loader)
print(classification_report(y_true,y_pred,target_names=['other','code'],zero_division=0,digits=3))
ConfusionMatrixDisplay(confusion_matrix(y_true,y_pred),display_labels=['other','code']).plot(cmap='Blues',values_format='d')
plt.title('مصفوفة الالتباس — جزء الاختبار'); plt.show()

test_eval=test_df.reset_index(drop=True).copy()
y_opencv=(test_eval.opencv_score.to_numpy()>=0.40).astype(int)
print('CNN  | test accuracy:',round(accuracy_score(y_true,y_pred),3),'| F1(code):',round(f1_score(y_true,y_pred,zero_division=0),3))
print('OpenCV heuristic | test accuracy:',round(accuracy_score(y_true,y_opencv),3),'| F1(code):',round(f1_score(y_true,y_opencv,zero_division=0),3))

## 7. تتبع مشاهد الفيديو بالنموذج المحفوظ\n\nنطبّق المصنف على إطارات العينة ونرسم احتمال وجود شاشة كود على محور الزمن. التنعيم هنا للعرض فقط، ولا يغير التنبؤات الخام. هذا لا يستخرج النص أو يعيد بناء ملفات المشروع.

In [ ]:
all_loader=DataLoader(FrameDataset(labels_df.assign(label='other')),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
# تنبؤات لكل الصفوف المعلمة؛ إن رغبت بمسح كل الإطارات، نعيد بناء Dataset بتسميات مؤقتة.
all_frame=manifest.copy(); all_frame['label']='other'
scan_loader=DataLoader(FrameDataset(all_frame),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
_,_,all_prob=collect_predictions(scan_loader)
plot_df=manifest.copy(); plot_df['p_code']=all_prob
plot_df['p_code_smooth']=pd.Series(all_prob).rolling(5,center=True,min_periods=1).mean().to_numpy()
plt.figure(figsize=(15,4)); plt.plot(plot_df.timestamp_sec/60,plot_df.p_code,alpha=.35,label='احتمال خام'); plt.plot(plot_df.timestamp_sec/60,plot_df.p_code_smooth,label='متوسط متحرك 5 عينات')
plt.axhline(.5,color='red',linestyle='--',label='عتبة 0.5'); plt.xlabel('الزمن بالدقائق'); plt.ylabel('احتمال code'); plt.ylim(0,1); plt.title('تتبّع ظهور شاشة الكود عبر الفيديو'); plt.grid(alpha=.25); plt.legend(); plt.show()
plot_df[['timestamp_sec','opencv_score','p_code','p_code_smooth']].to_csv(WORK_DIR/'video_timeline_predictions.csv',index=False)

## 8. حفظ النموذج والنتائج\n\nنحفظ أوزان `state_dict` مع تعريف الفئات وأبعاد الإدخال والتاريخ. الملف المحفوظ هو نموذج PyTorch حقيقي قابل لإعادة التحميل، بخلاف الملف النصي الوهمي الذي تنشئه خدمة التدريب الحالية. في Colab نزّل ملف النتائج من الخلية الأخيرة.\n\n### حدود التجربة\n\n- فيديو واحد لا يمثل محررات أو ألوانًا أو دقة شاشة مختلفة؛ يلزم فيديوهات متنوعة قبل تعميم النموذج.\n- الملصقات يدوية؛ خطؤها ينعكس على المقاييس.\n- مصنف CNN يكتشف وجود شاشة كود على مستوى الإطار، ولا يتعرف على نص الكود. استخراج النص يحتاج OCR وتقييمًا خاصًا مثل Character Error Rate ومراجعة ملفات المصدر.\n- النجاح هنا هو تدريب حقيقي ومقاييس اختبار قابلة لإعادة الإنتاج، وليس ضمان نجاح Studio بأكمله.

In [ ]:
checkpoint = {
    'state_dict':model.state_dict(),
    'class_to_idx':{'other':0,'code':1},
    'image_size':(90,160),
    'video_name':VIDEO_PATH.name,
    'sample_every_seconds':SAMPLE_EVERY_SEC,
    'seed':SEED,
    'best_validation_loss':float(best_val),
    'test_accuracy':float(accuracy_score(y_true,y_pred)),
    'test_f1_code':float(f1_score(y_true,y_pred,zero_division=0)),
}
model_path=WORK_DIR/'code_frame_cnn.pt'
torch.save(checkpoint,model_path)
summary={**{k:v for k,v in checkpoint.items() if k!='state_dict'},
         'epochs_completed':len(history_df),'training_seconds':float(history_df.seconds.iloc[-1]),
         'labels_used':len(labels_df),'test_samples':len(test_df)}
(WORK_DIR/'experiment_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))

try:
    from google.colab import files
    files.download(str(model_path)); files.download(str(WORK_DIR/'training_history.csv')); files.download(str(WORK_DIR/'video_timeline_predictions.csv'))
except ImportError:
    print('النموذج والسجلات محفوظة في:',WORK_DIR)